# AppleCiDEr · active learning with astronomical transients
### A CPU-only laboratory: interactive acquisition + a real research replay

**Two complementary investigations:**

- **Part A — Do it yourself:** train a lightweight classifier on compact astronomical features, query a simulated oracle, and compare random, entropy, margin and entropy-plus-diversity acquisition.
- **Part B — Understand AstroMiNN:** analyze saved *real* historical experiment results. These are measurements from completed training runs—not a model that can be retrained from CSVs.

Everything runs with NumPy, scikit-learn, Matplotlib and `al_pipeline`. No GPU, pretrained checkpoint, Felipe's model code, network connection or full ZTF archive is needed.

> **Data provenance is essential.** On a fresh clone, Part A uses an **explicitly synthetic** illustration, not real ZTF data. Once the curated real export is added to `examples/data/astro_real_v1.npz`, the *same notebook* switches to real observations automatically. Part B includes only a small, source-attributed set of actual presentation statistics until we add the audited historical CSV exports. Do not interpret synthetic demonstration learning curves as astronomical research results.

**Guiding question:** Which astronomical objects would you spend a limited spectroscopic follow-up budget on, and how does that choice change the final classifier?

## 1 · Environment and data origin

Run from a checkout of this repository after `python -m pip install -e ".[teaching]"`. All data used by the notebook is read locally. Set `USE_PRESENTATION_STYLE=False` to return to Matplotlib defaults.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display, Markdown
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score

from al_pipeline.astronomy.teaching_data import CLASS_NAMES, make_synthetic_demo, load_teaching_data
from al_pipeline.core.batch_selection import DiversitySelector
from al_pipeline.core.evaluation.history import EvaluationHistory
from al_pipeline.core.evaluation.metrics import compute_classification_metrics
from al_pipeline.core.evaluation.plotting import plot_strategy_comparison
from al_pipeline.core.evaluation.notebook_style import use_notebook_style, policy_color, polish_policy_axes
from al_pipeline.core.loop import ActiveLearningLoop
from al_pipeline.core.oracle.existing_labels import ExistingLabelsOracle
from al_pipeline.core.pool import Pool
from al_pipeline.core.scoring.random import RandomScorer
from al_pipeline.core.scoring.entropy import EntropyScorer
from al_pipeline.core.scoring.margin import MarginScorer
from al_pipeline.core.selection import TopKSelector
from al_pipeline.core.types import ClassificationPredictions

# Embed figures in GitHub's notebook preview, including when executing headlessly.
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
USE_PRESENTATION_STYLE = True
if USE_PRESENTATION_STYLE:
    use_notebook_style()

# Supports both starting Jupyter in the repository root and in examples/.
DATA_DIR = next((d for d in (Path('examples/data'), Path('data')) if d.is_dir()), Path('examples/data'))
REAL_DATA = DATA_DIR / 'astro_real_v1.npz'
SEED = 135
INITIAL_PER_CLASS = 4
BATCH_SIZE = 20
N_CYCLES = 6
POLICIES = ('random', 'entropy', 'margin', 'entropy_diversity')
METRICS = ('f1_macro', 'balanced_accuracy')

if REAL_DATA.exists():
    dataset = load_teaching_data(REAL_DATA)
    REAL_ASTRONOMY = True
else:
    dataset = make_synthetic_demo(seed=SEED)
    REAL_ASTRONOMY = False

banner = ('REAL ASTRONOMICAL EXPORT' if REAL_ASTRONOMY else
          '⚠ SYNTHETIC DEMONSTRATION — NOT ZTF DATA / NOT ASTROMINN RESULTS')
display(Markdown('**' + banner + '**'))
print('Feature provenance:', str(dataset['provenance']))
print('Representation:', str(dataset['feature_source']))

### What is in our object-level export?

The original historical data can contain many alerts for one astrophysical object. We acquire **objects**, not individual alert rows. The compact export keeps a representation of each object's latest available alert, its alert count and the original disjoint train/validation/test splits.

The **real export**, when installed, currently uses deterministic $8\times8$ image summaries from the three cutout channels plus 34 metadata values. These are **not** frozen AstroMiNN embeddings; we chose this checkpoint-free baseline first. A later, verified $L_0$-trained feature export could replace the representation without changing the notebook.

Class labels are available to the *simulated* oracle, but must never be passed to the acquisition policy before an object is selected.

In [ ]:
X = dataset['features']
y = dataset['labels']  # used only for partitioning, evaluation and a simulated oracle
object_ids = dataset['object_ids']
splits = dataset['splits']
alert_counts = dataset['alert_counts']
rows = {name: np.flatnonzero(splits == name) for name in ('train','validation','test')}
id_to_row = {str(oid): int(i) for i, oid in enumerate(object_ids)}
assert len(rows['train']) + len(rows['validation']) + len(rows['test']) == len(y)
assert len(set(object_ids)) == len(object_ids)
assert set(np.unique(y[rows['train']])) == set(range(len(CLASS_NAMES)))

split_table = pd.DataFrame({name: np.bincount(y[indices], minlength=len(CLASS_NAMES))
                           for name, indices in rows.items()}, index=CLASS_NAMES)
display(split_table.rename_axis('Target class'))
print('Objects:',len(y),' | feature dimension:',X.shape[1],
      '| median alerts/object:',int(np.median(alert_counts)))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.4))
axes[0].bar(CLASS_NAMES, split_table['validation'], color=policy_color('entropy'))
axes[0].set(title='Class support in the fixed validation set', ylabel='Objects')
axes[1].hist(alert_counts[rows['train']], bins=np.arange(1, 13)-0.5,
             color=policy_color('margin'), edgecolor='white')
axes[1].set(title='Available alert counts (training population)', xlabel='Alerts per object', ylabel='Objects')
for ax in axes: ax.grid(axis='y', alpha=.35)
plt.tight_layout()
plt.show()

## 2 · Meet the images

The cutouts contain the **science**, **reference** and **difference** channels. We show a few examples from the fixed validation set; the displayed labels are *for illustration only* and never become candidate labels in the active learning loop.

For a synthetic run, these are deliberately artificial Gaussian stamps. For a real export, they are subsampled channels of the actual historical `.npy` cutouts.

In [ ]:
thumbs = dataset['preview_images']
preview_ix = dataset['preview_indices']
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(7.8, 9))
for class_index, name in enumerate(CLASS_NAMES):
    matches = [j for j, idx in enumerate(preview_ix) if int(y[idx]) == class_index]
    if not matches: continue
    sample = thumbs[matches[0]]
    for channel, ax in enumerate(axes[class_index]):
        v = sample[channel]
        p1, p99 = np.percentile(v, [2, 98])
        ax.imshow(v, cmap='gray_r', vmin=p1, vmax=max(p99, p1 + .001))
        ax.set_xticks([]); ax.set_yticks([])
        if channel == 0: ax.set_ylabel(name, rotation=0, labelpad=26, va='center')
        if class_index == 0: ax.set_title(('Science', 'Reference', 'Difference')[channel])
fig.suptitle('Sample cutout triplets — ' + ('historical images' if REAL_ASTRONOMY else 'SYNTHETIC illustration'))
plt.tight_layout()
plt.show()

## 3 · Start with $L_0$, then ask the oracle

We use four initially observed objects per class. This stratification is a **benchmark design choice**, not a realistic claim that astronomers already have this many TDE labels. It avoids an impossible five-class fit at cycle zero.

The candidate pool consists only of the historical **training** split. Validation stays fixed for all acquisition policies; test remains sealed until the very end. `ExistingLabelsOracle` mimics label acquisition without charging telescope time.

In [ ]:
train_rows = rows['train']
rng = np.random.default_rng(SEED)
initial_rows = np.concatenate([
    rng.choice(train_rows[y[train_rows] == cls], size=INITIAL_PER_CLASS, replace=False)
    for cls in range(len(CLASS_NAMES))
])
initial_ids = tuple(str(object_ids[i]) for i in initial_rows)
initial_labels = {str(object_ids[i]): int(y[i]) for i in initial_rows}
train_ids = tuple(str(object_ids[i]) for i in train_rows)
# The following labels belong to the *oracle*, not the acquisition model.
oracle = ExistingLabelsOracle({str(object_ids[i]): int(y[i]) for i in train_rows})
assert len(initial_ids) == len(CLASS_NAMES)*INITIAL_PER_CLASS == 20
assert set(initial_ids).issubset(set(train_ids))
print('L0 class mix:', {CLASS_NAMES[c]: list(initial_labels.values()).count(c)
                         for c in range(len(CLASS_NAMES))})
print('Candidate pool size:', len(train_ids)-len(initial_ids))

### The model is deliberately small

A `StandardScaler` and multinomial `LogisticRegression` are trained **from scratch at every acquisition cycle** using only the currently revealed labels. Scaling is fitted on $L_t$ alone. This is a scientifically modest teaching surrogate, not the published AstroMiNN architecture.

In [ ]:
def fit_classifier(observed):
    indices = [id_to_row[oid] for oid in sorted(observed)]
    labels = np.asarray([observed[oid] for oid in sorted(observed)], dtype=int)
    assert set(np.unique(labels)) == set(range(len(CLASS_NAMES)))
    model = make_pipeline(StandardScaler(), LogisticRegression(
        C=0.15, max_iter=250, solver='lbfgs', class_weight='balanced'))
    model.fit(X[indices], labels)
    return model


def validation_scores(model):
    pred = ClassificationPredictions(model.predict_proba(X[rows['validation']]),
                                     output_type='probabilities')
    return compute_classification_metrics(y[rows['validation']], pred,
                                          metrics=METRICS, labels=tuple(range(len(CLASS_NAMES))))

baseline = fit_classifier(initial_labels)
display(pd.DataFrame([validation_scores(baseline)]).round(3))

## 4 · Put the active learning machinery to work

The same core interfaces used in the digits lab apply here. Random, entropy, margin and entropy-plus-diversity receive identical $L_0$, candidate IDs, and per-cycle budgets.

The diversity heuristic first takes a shortlist of high-entropy candidates, then spreads the selected batch across feature space. This is **within-batch** diversity, not full core-set coverage. The observed class of a newly acquired object is revealed *after* selection.

In [ ]:
def run_policy(policy, *, seed=SEED):
    observed = dict(initial_labels)
    pool = Pool.from_ids(train_ids, labeled_ids=initial_ids)
    history = EvaluationHistory()
    acquired_batches, acquired_labels = [], []
    generator = np.random.default_rng(seed)
    scorer = {'random': RandomScorer(generator), 'entropy': EntropyScorer(),
              'margin': MarginScorer(), 'entropy_diversity': EntropyScorer()}[policy]
    for cycle in range(N_CYCLES + 1):
        assert set(observed) == set(pool.labeled_ids)
        model = fit_classifier(observed)
        measures = validation_scores(model)
        history.add(cycle=cycle, n_labeled=len(observed),
                    n_acquired=0 if cycle == 0 else len(acquired_batches[-1]),
                    split='validation', metrics=measures,
                    strategy=policy, seed=seed, run_id=f'{policy}-{seed}')
        if cycle == N_CYCLES or not pool.unlabeled_ids:
            break
        candidates = sorted(pool.unlabeled_ids)
        candidate_rows = np.asarray([id_to_row[oid] for oid in candidates], dtype=int)
        predictions = (len(candidates) if policy == 'random' else
                       ClassificationPredictions(model.predict_proba(X[candidate_rows]),
                                                 output_type='probabilities'))
        selector = (DiversitySelector(X[candidate_rows], shortlist_multiplier=3.0)
                    if policy == 'entropy_diversity' else TopKSelector())
        selection = ActiveLearningLoop(scorer=scorer, selector=selector, pool=pool).acquire(
            ids=candidates, acquisition_data=predictions, k=min(BATCH_SIZE, len(candidates)))
        assert set(selection.ids).issubset(pool.pending_ids)
        assert not set(selection.ids).intersection(observed)
        received = oracle.query(selection)
        for oid, label in zip(received.ids, received.labels):
            observed[oid] = int(label)
        pool.resolve(received.ids)
        assert not pool.pending_ids
        acquired_batches.append(tuple(selection.ids))
        acquired_labels.append(tuple(int(z) for z in received.labels))
    return {'history':history, 'model':model, 'observed':observed, 'pool':pool,
            'batches':acquired_batches, 'revealed':acquired_labels}

runs = {policy: run_policy(policy) for policy in POLICIES}
expected = len(initial_ids) + N_CYCLES * BATCH_SIZE
assert all(run['history'].records[-1].n_labeled == expected for run in runs.values())
assert all(not run['pool'].pending_ids for run in runs.values())
summary = pd.DataFrame([{
    'policy': policy,
    'initial macro-F1': run['history'].records[0].metrics['f1_macro'],
    'final macro-F1': run['history'].records[-1].metrics['f1_macro'],
    'final balanced accuracy': run['history'].records[-1].metrics['balanced_accuracy'],
    'new labels': expected-len(initial_ids)
} for policy, run in runs.items()])
display(summary.round(3))
print('Same initial L0, same budgets, separate pool/oracle transitions for every policy.')

## 5 · Label efficiency and rare-class recovery

Always compare policy performance against the **number of labeled objects**, not just cycle number. A high final macro-F1 can conceal an inefficient early trajectory. In highly imbalanced settings, acquired class composition and rare-class recall matter just as much.

The acquired class labels shown below come from the oracle **after** acquisition. No policy sees them beforehand.

In [ ]:
combined = EvaluationHistory([record for run in runs.values()
                              for record in run['history'].records])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, METRICS, ('Validation macro-F1','Validation balanced accuracy')):
    plot_strategy_comparison(combined, metric, strategies=POLICIES,
                             split='validation', x_axis='n_labeled', ax=ax)
    ax.set_title(title); ax.set_ylabel(title)
    polish_policy_axes(ax)
fig.suptitle('Interactive CPU classifier — ' + ('real astronomical sample' if REAL_ASTRONOMY else 'SYNTHETIC EXAMPLE'))
plt.tight_layout()
plt.show()

display(pd.DataFrame([{
    'policy': policy, 'cycle': record.cycle, 'n_labeled': record.n_labeled,
    'macro_F1': record.metrics['f1_macro']}
    for policy, run in runs.items() for record in run['history'].records
]).pivot(index='n_labeled', columns='policy', values='macro_F1').round(3))

In [ ]:
fig, axes = plt.subplots(1, len(POLICIES), figsize=(13.2, 3.3), sharey=True)
for ax, policy in zip(axes, POLICIES):
    labels = [c for batch in runs[policy]['revealed'] for c in batch]
    hist = np.bincount(labels, minlength=len(CLASS_NAMES))
    ax.bar(range(len(CLASS_NAMES)), hist, color=policy_color(policy), edgecolor='white')
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
    ax.set_title(policy.replace('_',' + ')); ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('Revealed acquisition labels')
fig.suptitle('Class mixture of selected follow-up objects')
plt.tight_layout()
plt.show()

acquired = {p:set(oid for batch in run['batches'] for oid in batch)
            for p, run in runs.items()}
intersection = pd.DataFrame([
    [len(acquired[left] & acquired[right]) for right in POLICIES]
    for left in POLICIES], index=POLICIES, columns=POLICIES)
display(intersection)
print('Acquisition overlap counts exclude L0. Shared objects do not imply identical learning curves.')

### Who actually improves the rare classes?

The validation split contains labeled objects, so it is legitimate to compute **per-class recall** here. This is an evaluation, not information available to the acquisition policy. With small supports, class metrics can be noisy; interpret them alongside the counts.

In [ ]:
recall_rows = []
for policy, run in runs.items():
    pred = run['model'].predict(X[rows['validation']])
    recall = recall_score(y[rows['validation']], pred,
                          labels=np.arange(len(CLASS_NAMES)), average=None, zero_division=0)
    for label, score in zip(CLASS_NAMES, recall):
        recall_rows.append({'policy':policy, 'class':label, 'recall':float(score)})
recall_df = pd.DataFrame(recall_rows)
display(recall_df.pivot(index='class', columns='policy', values='recall').round(3))
fig, ax = plt.subplots(figsize=(9.0, 3.8))
for j, policy in enumerate(POLICIES):
    vals = recall_df[recall_df.policy == policy]['recall'].values
    ax.bar(np.arange(len(CLASS_NAMES)) + (j-(len(POLICIES)-1)/2)*.18,
           vals, width=.18, color=policy_color(policy), label=policy)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set(ylim=(0,1), ylabel='Validation recall', title='Per-class recall of the final CPU classifiers')
ax.legend(ncol=2); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## 6 · Plan B: what did the *actual* AstroMiNN experiments show?

The next comparison is **not** the synthetic fallback above. It is a small excerpt of independently measured, real historical experiments from the supplied AppleCiDEr research presentation (November 2026, acquisition-policy summary / rare-class performance slides).

In these runs the classifier was AstroMiNN, not the logistic-regression surrogate. **We cannot infer unseen retraining results from a saved CSV**, and metrics from the two models are **not directly comparable** because their datasets, training budgets, and representations differ.

A complete historical replay will automatically appear if the audited `astro_historical_evaluation.csv` export is placed in `examples/data/`. Until then, only the following carefully labeled *reported values* are available.

In [ ]:
# Reported point estimates from the presentation; NOT reconstructed learning curves.
reported = pd.DataFrame([
    {'model':'Full-data AstroMiNN', 'training fraction':1.0, 'macro-F1':.6367,
     'TDE F1':.194, 'TDE recall':.706, 'TDE false positives':95.7},
    {'model':'BALD at cycle 2', 'training fraction':.30, 'macro-F1':.6534,
     'TDE F1':.265, 'TDE recall':.510, 'TDE false positives':41.0},
    {'model':'TAILOR at cycle 2', 'training fraction':.30, 'macro-F1':.6516,
     'TDE F1':.288, 'TDE recall':.627, 'TDE false positives':46.7},
])
display(reported)
fig, axes = plt.subplots(1,2,figsize=(12,3.7))
bar_colors = ['#B7A588', '#83B6CD', '#AE9DC7']
for ax, metric, title in zip(axes, ('macro-F1','TDE F1'),
                             ('Validation macro-F1', 'TDE F1 (small support)')):
    ax.bar(reported.model, 100*reported[metric], color=bar_colors, width=.58)
    ax.set(title=title, ylabel='Percent')
    ax.tick_params(axis='x', labelrotation=18)
    ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()
print('Source: supplied Final.pdf, acquisition-policy findings and rare-class performance slides.')
print('These are *different* model-training recipes and labeled fractions, not a causal comparison.')

### Optional: full archived trajectories and acquisition composition

The historical reporting package can write complete per-seed tables. A one-time, small CPU export keeps only validation metrics and class/acquisition summaries, with no historical `.npy` files or checkpoints. The following cells **never invent missing cycles**: if no such export is present, they explain what is missing.

Required files (when reviewed and approved):
- `examples/data/astro_historical_evaluation.csv` — validation macro-F1 and balanced accuracy per policy, seed, cycle.
- Optional `astro_historical_acquisition.csv`, `astro_historical_overlap.csv`, `astro_historical_per_class.csv` — additional audits.

In [ ]:
real_replay_path = DATA_DIR / 'astro_historical_evaluation.csv'
if not real_replay_path.exists():
    print('Full historical replay not yet distributed. The table above is genuine but limited to reported points.')
    print('After a Delta export, replace this message with complete multi-seed AstroMiNN learning curves.')
    historical_df = None
else:
    historical_df = pd.read_csv(real_replay_path)
    required = {'suite','initialization','strategy','seed','cycle','split',
                'metric','value','n_labeled','n_acquired_total'}
    assert required.issubset(historical_df.columns), 'Historical export missing required columns'
    historical_df = historical_df[
        (historical_df['split'] == 'validation') &
        (historical_df['metric'].isin(['f1_macro','balanced_accuracy']))].copy()
    assert len(historical_df), 'Export lacks validation macro-F1/balanced accuracy'
    assert not historical_df.duplicated(['suite','initialization','strategy','seed','cycle','metric']).any()
    # Explicit choice avoids silently merging incompatible experiment suites.
    selected_suite = sorted(historical_df['suite'].unique())[0]
    selected_init = sorted(historical_df[historical_df.suite == selected_suite]['initialization'].unique())[0]
    subset = historical_df[(historical_df.suite == selected_suite) &
                           (historical_df.initialization == selected_init) &
                           (historical_df.metric == 'f1_macro')]
    stats = (subset.groupby(['strategy','cycle'],as_index=False)
             .agg(macro_F1=('value','mean'), sd=('value','std'),
                  n_labeled=('n_labeled','mean'), n_seeds=('seed','nunique')))
    fig, ax = plt.subplots(figsize=(9.5,4.3))
    for policy, group in stats.groupby('strategy'):
        group = group.sort_values('n_labeled')
        x = group.n_labeled.to_numpy(dtype=float)
        avg = group.macro_F1.to_numpy(dtype=float)
        sd = group.sd.fillna(0).to_numpy(dtype=float)
        ax.plot(x,avg,marker='o',label=str(policy),color=policy_color(str(policy)))
        ax.fill_between(x,avg-sd,avg+sd,alpha=.15,color=policy_color(str(policy)))
    ax.set(title=f'Recorded AstroMiNN trajectories: {selected_suite} / {selected_init}',
           xlabel='Cumulative labeled objects', ylabel='Validation macro-F1')
    ax.legend(ncol=2); ax.grid(axis='y',alpha=.3)
    plt.tight_layout();plt.show()
    display(stats.round(3).head(18))
    print('These curves belong to completed AstroMiNN training runs, not the CPU model above.')

In [ ]:
if historical_df is not None:
    extra = DATA_DIR / 'astro_historical_acquisition.csv'
    if extra.exists():
        acquired_history = pd.read_csv(extra)
        subset = acquired_history[(acquired_history.suite == selected_suite) &
                                  (acquired_history.initialization == selected_init)]
        last_cycle = subset['cycle'].max()
        final = subset[subset.cycle == last_cycle]
        composition = (final.groupby(['strategy','label'])['cumulative_count'].mean()
                       .unstack(fill_value=0))
        display(composition.round(1))
        ax = composition.plot(kind='bar',stacked=True,figsize=(9.0,3.6),colormap='Pastel1')
        ax.set(title=f'Historical acquired labels by cycle {last_cycle} (mean across seeds)',
               ylabel='Cumulative acquired objects',xlabel='Policy')
        ax.legend(title='Revealed class',bbox_to_anchor=(1,1))
        plt.tight_layout();plt.show()
    else:
        print('Optional acquisition-composition CSV not installed.')

## 7 · A fair conclusion, and your exercises

**What this notebook establishes:** a CPU-based closed loop can use *real* exported astronomical observations with unchanged general active-learning interfaces. The historical section describes separate AstroMiNN results, not how AstroMiNN would behave under new student-selected acquisitions.

**Try these optional exercises (left unsolved):**

1. **Alert availability:** compare the number of observed alerts for the random and uncertain acquired objects. Does the policy disproportionately target sources with many alerts? What would happen if eligibility allowed at most ten alerts?
2. **Rare class constraints:** compare the early-cycle acquisition counts of TDE and CV. Modify the selection policy to explore intentionally without reading hidden candidate labels.
3. **Oracle delays:** change the simulated oracle process so a few requests remain **pending** for a cycle. How should that affect the retraining set and acquisition budget?
4. **Representation swap:** once a frozen $L_0$-only AstroMiNN embedding export is available, rerun the CPU classifier and compare it with the simple pixel/metadata representation. Why would full-historical-data pretraining create leakage?
5. **Research replay:** once the full historical CSV export is installed, compare early area-under-the-learning-curve and class recovery across seeds. Can the policy with the best endpoint also be the least stable?

These exercises are intentionally open-ended. They should be attempted without using validation or hidden oracle labels to rank candidate objects.

## 8 · One sealed-test evaluation

For **Part A only**, pick a policy using *validation* macro-F1 and evaluate its final CPU classifier once on the fixed test split. This is a demonstration of separation of duties, not a published AppleCiDEr performance estimate. The historical AstroMiNN section never consults the notebook's test labels.

In [ ]:
winner = sorted(POLICIES, key=lambda p: (-runs[p]['history'].records[-1].metrics['f1_macro'], p))[0]
final_model = runs[winner]['model']
test_pred = ClassificationPredictions(final_model.predict_proba(X[rows['test']]),
                                      output_type='probabilities')
test_metrics = compute_classification_metrics(y[rows['test']], test_pred,
                                               metrics=METRICS,
                                               labels=tuple(range(len(CLASS_NAMES))))
print('Selected on validation:',winner)
print('ONE sealed-test score:',{key:round(value,3) for key,value in test_metrics.items()})
if not REAL_ASTRONOMY:
    print('Reminder: all Part A scores above come from synthetic class features, not ZTF.')